# Backtest Demo — S&P 500 1B log returns

Same flow as `getting_started/cpi_backtest_demo.ipynb`: **LastValuePredictor** vs **DartsAutoARIMAPredictor** only.

In [ ]:
import os
import sys
from pathlib import Path


def _bootstrap_sys_path() -> Path:
    if os.environ.get("AIENG_REPO_ROOT"):
        root = Path(os.environ["AIENG_REPO_ROOT"]).expanduser().resolve()
        if (root / "aieng-forecasting").is_dir():
            for sub in (root, root / "aieng-forecasting", root / "implementations"):
                s = str(sub)
                if s not in sys.path:
                    sys.path.insert(0, s)
            return root
    seeds: list[Path] = []
    try:
        from IPython import get_ipython  # type: ignore[import-not-found]

        ip = get_ipython()
        if ip is not None:
            nb = ip.user_ns.get("__vsc_ipynb_file__")
            if nb:
                seeds.append(Path(nb).resolve().parent)
    except Exception:
        pass
    for env_key in ("PWD", "INIT_CWD", "OLDPWD"):
        v = os.environ.get(env_key)
        if v:
            try:
                seeds.append(Path(v).expanduser().resolve())
            except (OSError, ValueError):
                pass
    try:
        seeds.append(Path.cwd().resolve())
    except (FileNotFoundError, OSError):
        pass
    for seed in seeds:
        for p in (seed, *seed.parents):
            if (p / "aieng-forecasting").is_dir() and (p / "implementations").is_dir():
                for sub in (p, p / "aieng-forecasting", p / "implementations"):
                    s = str(sub)
                    if s not in sys.path:
                        sys.path.insert(0, s)
                return p
    raise RuntimeError(
        "Cannot find repo root. Pick the project's .venv kernel, or set AIENG_REPO_ROOT."
    )


REPO_ROOT = _bootstrap_sys_path()

import aieng.forecasting  # noqa: F401
from aieng.forecasting.evaluation import BacktestSpec, backtest

import matplotlib.pyplot as plt
import pandas as pd
import yaml

from experiments.stock_price_forecasting_single_variable.data import (
    SP500_LOG_RETURN_SERIES_ID,
    build_sp500_log_return_service,
)
from methods.darts_arima import DartsAutoARIMAPredictor
from methods.naive import LastValuePredictor


## 1. Register log-return series

In [ ]:
svc = build_sp500_log_return_service(refresh=False, start="1990-01-01")
summary = svc.summary()
summary["start"] = summary["start"].dt.strftime("%Y-%m-%d")
summary["end"] = summary["end"].dt.strftime("%Y-%m-%d")
summary

## 2. Load spec

In [ ]:
spec_path = REPO_ROOT / "reference_specs" / "sp500_log_return_1b.yaml"
with spec_path.open() as f:
    spec = BacktestSpec.model_validate(yaml.safe_load(f))
print(spec.task.task_id, len(spec.origins()), spec.warmup)

## 3. Predictors

In [ ]:
naive_predictor = LastValuePredictor()
arima_predictor = DartsAutoARIMAPredictor(num_samples=500)

## 4. Run backtests

In [ ]:
naive_results = backtest(predictor=naive_predictor, spec=spec, data_service=svc)
arima_results = backtest(predictor=arima_predictor, spec=spec, data_service=svc)
for r in (naive_results, arima_results):
    print(r.predictor_id, len(r.predictions), r.mean_crps)

## 5. Per-origin CRPS

In [ ]:
def frame(result, label):
    return pd.DataFrame({
        "origin": [p.as_of.date() for p in result.predictions],
        "forecast_date": [p.forecast_date.date() for p in result.predictions],
        f"point_{label}": [p.payload.point_forecast for p in result.predictions],
        f"crps_{label}": result.scores,
    }).set_index("forecast_date")

naive_df = frame(naive_results, "naive")
arima_df = frame(arima_results, "arima")
comparison = naive_df.join(arima_df[["point_arima", "crps_arima"]], how="inner")
comparison.head(10)

## 6. Plot

In [ ]:
target_df = svc.get_series(SP500_LOG_RETURN_SERIES_ID, as_of=pd.Timestamp("2100-01-01"))
target_df = target_df.set_index("timestamp").sort_index()
win = target_df.loc[pd.Timestamp(spec.start) : pd.Timestamp(spec.end)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax1.plot(win.index, win["value"], color="black", linewidth=1, label="Observed")
ad = [p.forecast_date for p in arima_results.predictions]
ax1.plot(ad, [p.payload.point_forecast for p in arima_results.predictions], color="tab:blue", label="AutoARIMA")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax2.plot(ad, naive_results.scores, color="tab:orange", label="Naive CRPS")
ax2.plot(ad, arima_results.scores, color="tab:blue", label="ARIMA CRPS")
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()